# Logistic Regression: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains and tunes a logistic regression to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. It uses the same data, inputs and time-based folds as the other model notebooks, so they can be compared directly. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, following the EDA's decisions
3. trains a first logistic regression and checks it on the calibration split
4. tunes its settings with a search over time-based folds
5. scores the tuned model once on the calibration split and saves its predictions

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:** the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`, `SimpleImputer`, `StandardScaler`), the model (`LogisticRegression`), the tuning tool (`GridSearchCV`) and the scoring metrics
- **time / joblib:** timing the search and saving the models to files

In [1]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [2]:
train = pd.read_parquet("../../data/modeling/modeling_train.parquet")
calib = pd.read_parquet("../../data/modeling/modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

Training shape: (169827, 93)
Calibration shape: (297009, 93)


## 3. Building the Model Inputs
Uses the same inputs as the decision tree, random forest, SVM and XGBoost notebooks, so every model sees the same columns. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [3]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `lr_preprocess`:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers:**
     - fills blanks with the **training** median and adds a "was missing" column for each column that had blanks, the same as the tree models. The blanks come from specific drive models, such as S197 on two Toshiba models, so the fact that a value is missing is itself useful.
     - applies a signed log, `sign(x)·log(1+|x|)`. Raw counters span many orders of magnitude (power-on hours in the tens of thousands next to sector counts in single digits), and a few huge values would otherwise dominate a linear model. Trees don't need this because they only compare values against cutoffs. The sign is kept because changes can be negative.
     - scales every column to mean 0 and standard deviation 1 (`StandardScaler`). Logistic regression penalizes all coefficients by the same amount, so the columns must be on the same scale for that penalty to be fair.
   - **vendor:** one-hot encoded

Because this sits inside a pipeline, the medians, log and scaling are learned from training data only and re-used on the calibration split, so no information leaks from calibration into training.

In [4]:
def signed_log(X):
    return np.sign(X) * np.log1p(np.abs(X))

lr_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("log", FunctionTransformer(signed_log)),
            ("scale", StandardScaler()),
        ]), NUMERIC),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First Logistic Regression (Before Tuning)
Builds the full model, preprocessing followed by a logistic regression, with standard starting settings:

- **`C=1.0`:** the default regularization strength. Smaller values shrink the coefficients harder; larger values let them fit the training data more closely.
- **`penalty="l2"`, `solver="lbfgs"`:** the defaults. L2 shrinks every coefficient towards zero without removing any.
- **`class_weight="balanced"`:** gives failures more weight, the same as the tree models
- **`max_iter=5000`:** enough steps for the solver to converge
- **`random_state=42`:** makes results repeatable

In [5]:
lr_model = Pipeline([
    ("preprocessing", lr_preprocess),
    ("classifier", LogisticRegression(
        C = 1.0,
        class_weight = 'balanced',
        max_iter = 5000,
        random_state = 42
    ))
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split: the preprocessing learns its medians and scaling, then the coefficients are fitted. The fitted model is saved to `lr_base.joblib`.

In [6]:
lr_model.fit(X_train_raw, y_train)
joblib.dump(lr_model, "../artifacts/lr_base.joblib")

/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


['../artifacts/lr_base.joblib']

### 5.2 Checking the First Model on the Calibration Split
Predicts a failure probability for every calibration window, then turns those probabilities into yes/no predictions at thresholds from 0.0 to 1.0. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall
- **Brier score:** how close the probabilities are to what actually happened (lower is better)

PR-AUC, ROC-AUC and the Brier score don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

**Result:**
- PR-AUC on calib is **0.0739**, about 67 times the 0.0011 base rate. For comparison, the untuned decision tree scored 0.0701 and the tuned tree 0.0736.
- ROC-AUC is 0.888. The model ranks failures well overall, but its top-ranked windows are still mostly healthy drives.
- At 0.5, recall is 0.70 and precision is 0.015, so about 1 in 65 flagged windows actually fails.
- The Brier score of 0.093 is high because `class_weight="balanced"` pushes the probabilities up. The scores are useful for ranking, not as true probabilities.

In [7]:
results=[]
y_prob = lr_model.predict_proba(X_calib_raw)[:, 1]

# Default decision threshold
for x in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_pred = (y_prob >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "Logistic Regression",
        "PR-AUC (AP)": average_precision_score(y_calib, y_prob),
        "ROC-AUC": roc_auc_score(y_calib, y_prob),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_calib, y_prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1,Brier Score
0,0.0,Logistic Regression,0.0739,0.8877,0.0011,1.0000,0.0022,0.0927
1,0.1,Logistic Regression,0.0739,0.8877,0.0011,1.0000,0.0022,0.0927
2,0.2,Logistic Regression,0.0739,0.8877,0.0015,0.9543,0.0030,0.0927
3,0.3,Logistic Regression,0.0739,0.8877,0.0036,0.8415,0.0072,0.0927
4,0.4,Logistic Regression,0.0739,0.8877,0.0100,0.7530,0.0198,0.0927
5,0.5,Logistic Regression,0.0739,0.8877,0.0153,0.7012,0.0299,0.0927
6,0.6,Logistic Regression,0.0739,0.8877,0.0189,0.6585,0.0368,0.0927
7,0.7,Logistic Regression,0.0739,0.8877,0.0231,0.6311,0.0447,0.0927
8,0.8,Logistic Regression,0.0739,0.8877,0.0284,0.5793,0.0542,0.0927
9,0.9,Logistic Regression,0.0739,0.8877,0.0379,0.5396,0.0708,0.0927


## 6. Tuning the Logistic Regression
### 6.1 Search Setup and Time-Based Folds
Sets up the tuning:

- **`lr_model`:** the search reuses the model from section 5. It works on a fresh copy, so the first model is left untouched. `class_weight="balanced"` and `random_state=42` carry over to every combination.
- **`param_grid`:** the settings to try:
  - regularization strength `C`: 0.001 to 100, in steps of 10
  - penalty and solver together, since not every solver supports every penalty:
    - **L2** (shrinks all coefficients) with `lbfgs`, `liblinear` and `saga`
    - **L1** (can set coefficients to exactly zero, dropping weak columns) with `liblinear` and `saga`
    - **elastic net** (half L1, half L2) with `saga`, the only solver that supports it

  Together that's 36 combinations. The grid is small enough to try every one, so this uses a full grid search rather than a random sample.
- **`time_folds()`:** builds 4 cross-validation folds **by time**, the same function as every other model notebook. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead.

In [8]:
C_VALUES = [0.001, 0.01, 0.1, 1, 10, 100]

param_grid = [
    {"classifier__C": C_VALUES, "classifier__penalty": ["l2"], "classifier__solver": ["lbfgs", "liblinear", "saga"]},
    {"classifier__C": C_VALUES, "classifier__penalty": ["l1"], "classifier__solver": ["liblinear", "saga"]},
    {"classifier__C": C_VALUES, "classifier__penalty": ["elasticnet"], "classifier__solver": ["saga"], "classifier__l1_ratio": [0.5]},
]


## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")

fold 0: train 28,508 rows -> validate 37,027 rows
fold 1: train 64,596 rows -> validate 31,106 rows
fold 2: train 95,671 rows -> validate 32,768 rows
fold 3: train 128,057 rows -> validate 33,501 rows


### 6.2 Untuned Model on the Time Folds
Scores the untuned logistic regression from section 5 on the same 4 time-based folds the search uses, with the same four metrics: precision, recall, F1 and PR-AUC. This is the "before" number for tuning.

**Why this is needed.** Section 5.2 scored the untuned model on the calibration split, where failures are about 1 in 905. The search scores on training folds, where they are 1 in 21. PR-AUC depends heavily on the failure rate, so the 5.2 score can't be compared with the search scores in 6.3. Scoring the untuned model on the same folds the search uses leaves tuning as the only difference between them.

- **`cross_validate`:** trains the untuned model once per fold, scores it on that fold's validation months, and reports the mean and std of each metric over the 4 folds.
- **`scoring`:** the same four metrics the search uses, so the before and after numbers are measured the same way.

**Result:**
- PR-AUC: **0.532** (std 0.015)
- Recall: 0.668 (std 0.028)
- Precision: 0.338 (std 0.034)
- F1: 0.448 (std 0.035)
- This is the baseline for the search. The untuned decision tree scored 0.527 on the same folds.

In [9]:
scoring={
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
            "pr_auc": "average_precision",
        }

cv_untuned_results = cross_validate(lr_model, X_train_raw, y_train, cv = cv_folds, scoring = scoring, n_jobs = -1)

for i in cv_untuned_results:
    print(f'{i} mean: {np.mean(cv_untuned_results[i])}')
    print(f'{i} standard deviation: {np.std(cv_untuned_results[i])}')
    print(' ')

/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


fit_time mean: 1.50821852684021
fit_time standard deviation: 0.7212588335535848
 
score_time mean: 0.19635170698165894
score_time standard deviation: 0.04013078386409318
 
test_precision mean: 0.3379935634060286
test_precision standard deviation: 0.03353379716328111
 
test_recall mean: 0.6682157844434444
test_recall standard deviation: 0.028227924525796266
 
test_f1 mean: 0.44842755463105366
test_f1 standard deviation: 0.034837847131602075
 
test_pr_auc mean: 0.5320024460175174
test_pr_auc standard deviation: 0.015370094532339494
 


### 6.3 Grid Search, Results and Saving the Best Model
- **Grid search:** tries all **36 combinations**. Each is trained once per fold, giving **144 fits**, and scored by PR-AUC (`average_precision`).
- **Refit:** the best combination is retrained on all of the training data (`refit="pr_auc"`).
- **Results table:**
  - `PR_AUC`: the average PR-AUC over the 4 folds
  - `PR_AUC_std`: how much the 4 fold scores vary
  - `PR_AUC_rank`: 1 = best

  The table is saved to `lr_search_results_ranked.csv`.
- **Save:** the best model is saved to `lr_best.joblib`, so it can be reloaded without retraining.

**Result:**
- Best: **C = 100, l2, liblinear**, with PR-AUC **0.5326** (std 0.015)
- The untuned model scored 0.5320, so tuning gained 0.0006, which is far smaller than the fold std of 0.015. Tuning made no real difference.
- The top 10 settings all fall between 0.5320 and 0.5326. Penalty and solver hardly matter once C ≥ 1.
- Strong regularisation hurts: every C = 0.001 setting ranks at the bottom (0.515–0.517).
- C = 100 sits on the edge of the grid. The scores are flat from C = 1 upward, so a larger C would not change the result.
- Some `saga` fits stopped at `max_iter` without converging, and they ran slowest (up to about 300 s per fold). They don't rank above the `liblinear` and `lbfgs` fits.

These scores come from training folds, where failures are 1 in 21, so they're much higher than PR-AUC on the calibration split. Use them only to compare settings with each other and with section 6.2.

In [10]:
search = GridSearchCV(
    estimator=lr_model,
    param_grid=param_grid,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "pr_auc": "average_precision",
    },
    cv=cv_folds,
    n_jobs=-1,
    refit="pr_auc",
    verbose=1,
    return_train_score=False,
)


start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

cv_results = pd.DataFrame(search.cv_results_)

# All four metrics, averaged across the CV folds (column names follow the scoring keys)
results = pd.DataFrame({
    "PR_AUC":         cv_results["mean_test_pr_auc"],
    "PR_AUC_std":     cv_results["std_test_pr_auc"],
    "Precision":      cv_results["mean_test_precision"],
    "Precision_std":  cv_results["std_test_precision"],
    "Recall":         cv_results["mean_test_recall"],
    "Recall_std":     cv_results["std_test_recall"],
    "F1":             cv_results["mean_test_f1"],
    "F1_std":         cv_results["std_test_f1"],
    "mean_fit_time":  cv_results["mean_fit_time"],
})

for col in cv_results.columns:
    if col.startswith("param_"):
        results[col] = cv_results[col]

# Rank by PR-AUC (the same ranking the search used to pick the best model)
results["PR_AUC_rank"] = cv_results["rank_test_pr_auc"]
results = results.sort_values("PR_AUC_rank").reset_index(drop=True)

print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

print("\nTop 10 models ranked by PR-AUC:")
display(results.head(10))

results.to_csv("lr_search_results_ranked.csv", index=False)

# Save the best model (chosen by PR-AUC, refit on all of train)
best_lr = search.best_estimator_
joblib.dump(best_lr, "../artifacts/lr_best.joblib")

Fitting 4 folds for each of 36 candidates, totalling 144 fits


/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


... (37 outputs left out)

/home/user/IST-707-Group-Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Search took 88.1 minutes
Best PR-AUC (time-CV on train): 0.5326
Best parameters:
  classifier__C: 100
  classifier__penalty: l2
  classifier__solver: liblinear

Top 10 models ranked by PR-AUC:


,PR_AUC,PR_AUC_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,mean_fit_time,param_classifier__C,param_classifier__penalty,param_classifier__solver,param_classifier__l1_ratio,PR_AUC_rank
0,0.532566,0.015322,0.338380,0.033606,0.668032,0.027478,0.448711,0.034711,4.044672,100.0,l2,liblinear,NaN,1
1,0.532546,0.015272,0.338479,0.033416,0.668032,0.027478,0.448806,0.034538,50.776808,100.0,l1,liblinear,NaN,2
2,0.532539,0.015274,0.338367,0.033542,0.668032,0.027478,0.448702,0.034658,53.169166,10.0,l1,liblinear,NaN,3
3,0.532534,0.015243,0.338708,0.033801,0.667845,0.027722,0.448958,0.034935,57.703820,1.0,l1,liblinear,NaN,4
4,0.532518,0.015275,0.338470,0.033592,0.668032,0.027478,0.448792,0.034706,3.322124,10.0,l2,liblinear,NaN,5
5,0.532400,0.015357,0.338259,0.033808,0.667550,0.028202,0.448503,0.035067,3.805981,1.0,l2,liblinear,NaN,6
6,0.532251,0.015516,0.338364,0.033878,0.667899,0.028007,0.448667,0.035078,297.875862,1.0,l2,saga,NaN,7
7,0.532233,0.015358,0.337489,0.033579,0.668510,0.027748,0.448047,0.034870,1.484325,10.0,l2,lbfgs,NaN,8
8,0.532214,0.015125,0.338010,0.032910,0.668161,0.027946,0.448459,0.034293,1.526515,100.0,l2,lbfgs,NaN,9
9,0.532002,0.015370,0.337994,0.033534,0.668216,0.028228,0.448428,0.034838,1.529961,1.0,l2,lbfgs,NaN,10


['../artifacts/lr_best.joblib']

## 7. Final Scoring on the Calibration Split
Scores the tuned logistic regression from 6.3 **once** on the calibration split and saves its predictions for the model comparison. Nothing is tuned here.

- **Why calib:** the search chose its settings on training folds, where failures are 1 in 21. The calibration split (November to December 2024) has failures at the real rate of about 1 in 905, comes after every training month, and was never used to fit or tune this model. Every model is scored on exactly the same rows, so the scores can be compared directly.
- **Why only once:** calib holds only 328 failures. Scoring it repeatedly and adjusting the model each time would tune the model to those 328 drives. The 2025 test split stays closed until the final model is chosen.
- **The model:** loaded from `../artifacts/lr_best.joblib`, the same file the final test will load.
- **Saved file:** `lr_calib_preds.csv` in this folder, one row per calibration window, with the columns `serial_number`, `window`, `label` and `prob`.

**Result:**
- PR-AUC on calib: **0.0750** (297,009 windows, 328 failures)
- The untuned model in 5.2 scored 0.0739. As in 6.3, the tuned and untuned models are essentially the same.
- The tuned decision tree scored 0.0736 on the same rows, so logistic regression is level with it.

In [11]:
lr_final = joblib.load("../artifacts/lr_best.joblib")

calib_scores = lr_final.predict_proba(X_calib_raw)[:, 1]

calib_preds = pd.DataFrame({
    'serial_number': meta_calib['serial_number'].copy(),
    'window': meta_calib['window'].copy(),
    'label': y_calib.to_numpy(),
    'prob': calib_scores
})

calib_preds.to_csv('lr_calib_preds.csv', index = False)

print(f"Average precision score: {average_precision_score(calib_preds['label'], calib_preds['prob'])}")
print(f'Row count: {len(calib_preds)}')
print(f"Failure count: {calib_preds['label'].sum()}")

Average precision score: 0.07497944127785963
Row count: 297009
Failure count: 328
